In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sqlite3
from estnltk import Text
from estnltk.taggers import VabamorfAnalyzer
import json
import csv
import pandas as pd
import random

### Setup

In [4]:
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
CONFLICTS = "conflict_results.db"

In [5]:
morph_analyzer = VabamorfAnalyzer(output_layer="morph_analysis")

In [6]:
# Lausest on vaja leida õige sõnavorm ja selle asukoht (span)

def find_word_form(phrase_root_loc: int, 
                   phrase_root_lemma: str, 
                   sentence: str) -> tuple[str, list[int, int]] | None:
    # VabamorfAnalyzer-ga tekkis sõnavormi otsingutel lemma alusel rohkem probleeme
    #sentence_text = Text(sentence).tag_layer(["words", "sentences"])
    #morph_analyzer.tag(sentence_text)
    sentence_text = Text(sentence).tag_layer("morph_analysis")
    i = phrase_root_loc-1

    word_form = sentence_text.morph_analysis[i]

    if phrase_root_lemma in word_form.lemma:
        return word_form.text, [word_form.start, word_form.end]
    else:
        # vaatame eelnevat ja järgnevat sõna, sest andmebaasist saadud lemma ei vasta alati morf analüüsi kihi segmentatsioonile
        previous_word_lemma = None
        next_word_lemma = None
        if i > 0:
            for lemma in sentence_text.morph_analysis[i-1].lemma:
                if lemma in phrase_root_lemma:
                    previous_word_lemma = lemma
                    break
        
        if i < len(sentence_text.morph_analysis)-1:
            for lemma in sentence_text.morph_analysis[i+1].lemma:
                if lemma in phrase_root_lemma:
                    next_word_lemma = lemma
                    break
        
        if phrase_root_lemma == previous_word_lemma:
            return sentence_text.morph_analysis[i-1].text, [sentence_text.morph_analysis[i-1].start, sentence_text.morph_analysis[i-1].end]
        elif phrase_root_lemma == next_word_lemma:
            return sentence_text.morph_analysis[i+1].text, [sentence_text.morph_analysis[i+1].start, sentence_text.morph_analysis[i+1].end]
        
        else:
            if previous_word_lemma and next_word_lemma:
                return sentence_text.morph_analysis[i-1].text + word_form.text + sentence_text.morph_analsysis[i+1].text, [sentence_text.morph_analysis[i-1].start, sentence_text.morph_analsysis[i+1].end]
            elif previous_word_lemma:
                return sentence_text.morph_analysis[i-1].text + word_form.text, [sentence_text.morph_analysis[i-1].start, word_form.end]
            elif next_word_lemma:
                word_form.text + sentence_text.morph_analysis[i+1].text, [word_form.start, sentence_text.morph_analysis[i+1].end]
            else:
                return None

### I Nsubj ^(nom|part)

In [7]:
# Loeme vajalikud andmed nsubj konfliktide tabelist (nsubj pol nominatiivis ega partitiivis)
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICTS}")
cur = con.cursor()

cur.execute("""
SELECT
    sentence_id,
    phrase_deprel,
    current_case,
    current_analysis,
    phrase_root_loc,
    phrase_root_lemma,
    sentence
FROM
    nsubj_not_nom_part
""")

result = cur.fetchall()

con.close()

In [8]:
result[:5]

[(12410790,
  'nsubj',
  'gen',
  'gen,prop,sg',
  1,
  'Kristina',
  'Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .'),
 (16715193,
  'nsubj',
  'gen',
  'gen,prop,sg',
  10,
  'Torupill',
  'Küpsetamist-kaunistamist jätkub tegelikult aasta lõpuni : sel pühapäeval aitab Torupilli Selveris piparkooke teha ja valmis piparkoogimaja detaile kokku panna pagar , järgmisel pühapäeval on tal kaubamaja kodumaailmas abiks Pipi .'),
 (214160,
  'nsubj',
  'gen',
  'com,gen,sg',
  5,
  'laen',
  'Kui 20. sajandi algul laenu- ja hoiuühistud aitasid vältida eestlaste hoiustevoolu Venemaale , siis 21. sajandi algul võiks neist abi olla hoiuste äravoolu takistamisel ääremaalt ettevõtluskeskustesse .'),
 (225510,
  'nsubj',
  'gen',
  'com,gen,sg',
  1,
  'tarkvara',
  'Tarkvara aitab jagu saada mitmes ametkonnas võidutsevast liigsest bürokraatiast .'),
 (325158,
  'nsubj',
  'gen',

#### Andmestik

Mida sisaldama hakkab:
- **sentence_id** lause ID koondkorpuses
- **sentence** lause tekstina
- **word_form** konfliktne sõnavorm tekstina (leitakse eraldi funktsiooni abil lausest, sest konfliktide andmebaasis on ainult lemma ja selle asukohaindeks lauses)
- **span** sõnavormi algus- ja lõpupositsioon lauses (listina)
- **current_deprel** konfliktse sõnavormi sõltuvussüntaktiline märgend hetkeseisuga (automaatsüntaksi määratud)
- **current_case** konfliktse sõnavormi ühestatud kääne hetkeseisuga (Vabamorfi määratud)
- **current_analysis** konfliktse sõnavormi kohta täiendav morfoinformatsioon, võibolla ei lähe tarvis, aga seal on näiteks informatsiooni, kas tegemist on näiteks pärisnimega jne, mida depreli ja käände teadmine meile üksi ei paku

### I nsubj

In [9]:
# Tekitame andmestiku JSON-i tarbeks
data = []
error_data = []

for idx, row in enumerate(result):
    try:
        form, span = find_word_form(row[4], row[5], row[6])
        if form and span:
            data.append({"sentence_id": row[0],
                     "sentence": row[6],
                     "word_form": form,
                     "span": span,
                     "current_deprel": row[1],
                     "current_case": row[2],
                     "current_analysis": row[3]})
    except:
        error_data.append([row[0], row[1], row[2], row[3], row[4], row[5], row[6]])
        print(idx, row)
    

1522 (12635873, 'nsubj', 'gen', 'gen,prop,sg', 2, 'Kose-Lükatile', 'Jääb Kose-Lükatile')


In [10]:
data[:5]

[{'sentence_id': 12410790,
  'sentence': 'Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .',
  'word_form': 'Kristina',
  'span': [0, 8],
  'current_deprel': 'nsubj',
  'current_case': 'gen',
  'current_analysis': 'gen,prop,sg'},
 {'sentence_id': 16715193,
  'sentence': 'Küpsetamist-kaunistamist jätkub tegelikult aasta lõpuni : sel pühapäeval aitab Torupilli Selveris piparkooke teha ja valmis piparkoogimaja detaile kokku panna pagar , järgmisel pühapäeval on tal kaubamaja kodumaailmas abiks Pipi .',
  'word_form': 'Torupilli',
  'span': [79, 88],
  'current_deprel': 'nsubj',
  'current_case': 'gen',
  'current_analysis': 'gen,prop,sg'},
 {'sentence_id': 214160,
  'sentence': 'Kui 20. sajandi algul laenu- ja hoiuühistud aitasid vältida eestlaste hoiustevoolu Venemaale , siis 21. sajandi algul võiks neist abi olla hoiuste äravoolu takistamisel ääremaalt ettevõtluskeskust

In [11]:
error_data

[[12635873,
  'nsubj',
  'gen',
  'gen,prop,sg',
  2,
  'Kose-Lükatile',
  'Jääb Kose-Lükatile']]

In [21]:
# Salvestame JSON-faili
DATA_DIR = "data/"
OUTFILENAME = "nsubj_conflicts_dataset_15022026.json"

with open(f"{DATA_DIR}{OUTFILENAME}", "w") as f:
    json.dump(data, f, indent=1, ensure_ascii=False)

In [22]:
# Salvestame vigased read CSV-sse

DATA_DIR = "data/"
ERROROUTFILENAME = "nsubj_conflicts_errors.csv"

header = ["sentence_id",
        "phrase_deprel",
        "current_case",
        "current_analysis",
        "phrase_root_loc",
        "phrase_root_lemma",
        "sentence"]

with open(f"{DATA_DIR}{ERROROUTFILENAME}", 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for row in error_data:
        writer.writerow(row)

In [23]:
# Loeme JSON-failist

with open(f"{DATA_DIR}{OUTFILENAME}", "r") as f:
    json_data = json.load(f)

In [24]:
json_data[:5]

[{'sentence_id': 12410790,
  'sentence': 'Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .',
  'word_form': 'Kristina',
  'span': [0, 8],
  'current_deprel': 'nsubj',
  'current_case': 'gen',
  'current_analysis': 'gen,prop,sg'},
 {'sentence_id': 16715193,
  'sentence': 'Küpsetamist-kaunistamist jätkub tegelikult aasta lõpuni : sel pühapäeval aitab Torupilli Selveris piparkooke teha ja valmis piparkoogimaja detaile kokku panna pagar , järgmisel pühapäeval on tal kaubamaja kodumaailmas abiks Pipi .',
  'word_form': 'Torupilli',
  'span': [79, 88],
  'current_deprel': 'nsubj',
  'current_case': 'gen',
  'current_analysis': 'gen,prop,sg'},
 {'sentence_id': 214160,
  'sentence': 'Kui 20. sajandi algul laenu- ja hoiuühistud aitasid vältida eestlaste hoiustevoolu Venemaale , siis 21. sajandi algul võiks neist abi olla hoiuste äravoolu takistamisel ääremaalt ettevõtluskeskust

In [12]:
# CSV-sse ka andmestik

csv_data = []

for dct in data:
    csv_data.append({
        'sentence_id': dct['sentence_id'],
        'start': dct['span'][0],
        'end': dct['span'][1],
        'word_form': dct['word_form'],
        'current_case': dct['current_case'],
        'sentence': dct['sentence'],
        'current_deprel': dct['current_deprel'],
        'current_analysis': dct['current_analysis']
    })

In [13]:
csv_data[:5]

[{'sentence_id': 12410790,
  'start': 0,
  'end': 8,
  'word_form': 'Kristina',
  'current_case': 'gen',
  'sentence': 'Kristina Shmiguni ahistas Holmenkollenis 30 km klassikatehnikasõidu lõpukilomeetritel kurnatus , ent viiendast kohast maailma karikasarja liidrirüü hoidmiseks piisas .',
  'current_deprel': 'nsubj',
  'current_analysis': 'gen,prop,sg'},
 {'sentence_id': 16715193,
  'start': 79,
  'end': 88,
  'word_form': 'Torupilli',
  'current_case': 'gen',
  'sentence': 'Küpsetamist-kaunistamist jätkub tegelikult aasta lõpuni : sel pühapäeval aitab Torupilli Selveris piparkooke teha ja valmis piparkoogimaja detaile kokku panna pagar , järgmisel pühapäeval on tal kaubamaja kodumaailmas abiks Pipi .',
  'current_deprel': 'nsubj',
  'current_analysis': 'gen,prop,sg'},
 {'sentence_id': 214160,
  'start': 22,
  'end': 28,
  'word_form': 'laenu-',
  'current_case': 'gen',
  'sentence': 'Kui 20. sajandi algul laenu- ja hoiuühistud aitasid vältida eestlaste hoiustevoolu Venemaale , siis 21

In [14]:
DATA_DIR = "data/"
CSV_OUTFILENAME = "nsubj_conflicts_dataset_15022026.csv"

fieldnames = ["sentence_id", "start", "end", "word_form", "current_case", "sentence", "current_deprel", "current_analysis"]

with open(f"{DATA_DIR}{CSV_OUTFILENAME}", mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(csv_data)

In [ ]:
# Väiksem näidisandmestik CSV-failist DataFrame-na, read valitakse juhuslikult

def create_example_dataframe(csv_filename: str, n_example_rows: int) -> pd.DataFrame:
    df = pd.read_csv(csv_filename, header=0)
    ids = [i for i in range(len(df))]
    random.shuffle(ids)

    return df.loc[ids[:n_example_rows]]

In [ ]:
df = create_example_dataframe(f"{DATA_DIR}{CSV_OUTFILENAME}", 25)

In [30]:
df

,sentence_id,sentence,word_form,start,end,current_deprel,current_case,current_analysis
1549,12168807,"Madalrõhkkond , mis Norra merelt kagusse liikudes tõmbas Skandinaavia- ja Läänemeremaadesse külma õhumassi , on jõudnud Kesk-Venemaale ja läheb meist üha kaugemale .",Kesk-Venemaale,120,134,nsubj,gen,"gen,prop,sg"
8817,2808920,"Ta tahab kellelegi tšekke ulatada , siis võtku oma isiklikult arvelt see raha , "" ütles Ansip .",oma,47,50,nsubj,gen,"gen,sg"
4324,16626522,keegi skoori ei oska õelda ?,skoori,6,12,nsubj,gen,"com,gen,sg"
274,10909047,Eile arutasid projekti linnavolikogu neli komisjoni .,komisjoni,42,51,nsubj,gen,"com,gen,sg"
3563,5144715,"President Carteri ajal justiitsministri kohal olnud Clarki nimetavad tema kritiseerijad Serbia vanameelsete pooldajaks , toetajad aga tunnustavad teda inimõiguste eest seisjana .",Clarki,52,58,nsubj,gen,"gen,prop,sg"
9340,18393733,"1993. aastal võttis Riigikogu vastu seaduse “ Eesti Vabariigi territooriumil endise NSV Liidu relvajõudude valduses või kasutuses olnud ja olevate maa-alade , hoonete ja rajatistega sooritatud tehingute kehtetuks tunnistamine ” .",Riigikogu,20,29,nsubj,gen,"com,gen,sg"
9438,8885003,"Joonis- ja nukufilmistuudiod on veetnud kollektiivpuhkusi , mõni üksik dokumentalist nokitseb kah oma “ koduvideot ” - sõna , millega heinakuises Postimehes võttis eesti tõsielufilmi hetketaseme kriitiliselt kokku Lauri Kärk .",Lauri,214,219,nsubj,gen,"gen,prop,sg"
4025,21039632,Dammu: ykski tydruk ei näita,ykski,7,12,nsubj,gen,"com,gen,sg"
4632,12381441,"Phüi , paneb Midri suu- ja südamepõhjast mätaste vahele , enne kui hakkab tagasi minema kalmuaia keskele , kus värskelt ülesloobitud mullakuhila man haigutab avatud kaevušaht nagu irvitav , inetu paise .",Midri,13,18,nsubj,gen,"gen,prop,sg"
2554,8121038,80 protsenti Estonijat ja Vestit kirjastavast firmast kuulub Siffile möödunud aasta novembrist .,Siffile,61,68,nsubj,gen,"gen,prop,sg"


In [31]:
# salvestame näidisandmed ka CSV-sse
EXAMPLE_OUTFILENAME = "nsubj_conflicts_dataset_15022026_example_25.csv"

df.to_csv(f"{DATA_DIR}{EXAMPLE_OUTFILENAME}", index=False)

### Ülejäänud

In [15]:
def get_data(CONFLICT_DB_PATH: str, conflict_table_name: str):
    con = sqlite3.connect(CONFLICT_DB_PATH)
    cur = con.cursor()

    cur.execute("""
    SELECT
        sentence_id,
        phrase_deprel,
        current_case,
        current_analysis,
        phrase_root_loc,
        phrase_root_lemma,
        sentence
    FROM
        {conflict_table_name}
    """.format(conflict_table_name=conflict_table_name))

    result = cur.fetchall()

    con.close()

    # Tekitame andmestiku JSON-i tarbeks
    data = []
    error_data = []

    for idx, row in enumerate(result):
        try:
            form, span = find_word_form(row[4], row[5], row[6])
            if form and span:
                data.append({"sentence_id": row[0],
                        "sentence": row[6],
                        "word_form": form,
                        "span": span,
                        "current_deprel": row[1],
                        "current_case": row[2],
                        "current_analysis": row[3]})
        except:
            error_data.append([row[0], row[1], row[2], row[3], row[4], row[5], row[6]])
            #print(idx, row)

    return data, error_data
    

In [18]:
def save_json_data_to_csv(json_data, OUTFILEPATH):
    # CSV-sse ka andmestik

    csv_data = []

    for dct in json_data:
        csv_data.append({
            'sentence_id': dct['sentence_id'],
            'start': dct['span'][0],
            'end': dct['span'][1],
            'word_form': dct['word_form'],
            'current_case': dct['current_case'],
            'sentence': dct['sentence'],    
            'current_deprel': dct['current_deprel'],
            'current_analysis': dct['current_analysis']
        })

    fieldnames = ["sentence_id", "start", "end", "word_form", "current_case", "sentence", "current_deprel", "current_analysis"]

    with open(OUTFILEPATH, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_data)

In [19]:
def create_example_dataframe(csv_infilename: str, n_example_rows: int, OUTFILEPATH: str):
    df = pd.read_csv(csv_infilename, header=0)
    ids = [i for i in range(len(df))]
    random.shuffle(ids)

    result_df = df.loc[ids[:n_example_rows]]

    result_df.to_csv(OUTFILEPATH, index=False)

In [20]:
create_example_dataframe(f"{DATA_DIR}nsubj_conflicts_dataset_15022026.csv", 25, f"{DATA_DIR}nsubj_conflicts_dataset_15022026_example_25.csv")

### nsubj:cop

In [14]:
nsubj_cop_data, nsubj_cop_error_data = get_data(f"{SOURCE_DATA_PATH}{CONFLICTS}", "nsubj_cop_not_nom_part")

# Salvestame JSON-faili
DATA_DIR = "data/"
OUTFILENAME = "nsubj_cop_conflicts_dataset_13032026.json"

with open(f"{DATA_DIR}{OUTFILENAME}", "w") as f:
    json.dump(nsubj_cop_data, f, indent=1, ensure_ascii=False)

In [22]:
len(nsubj_cop_error_data)

0

In [28]:
save_json_data_to_csv(nsubj_cop_data, f"{DATA_DIR}nsubj_cop_conflicts_dataset_13032026.csv")
create_example_dataframe(f"{DATA_DIR}nsubj_cop_conflicts_dataset_13032026.csv", 25, f"{DATA_DIR}nsubj_cop_conflicts_dataset_13032026_example_25.csv")

### obj

In [16]:
obj_data, obj_error_data = get_data(f"{SOURCE_DATA_PATH}{CONFLICTS}", "obj_not_nom_gen_part")

# Salvestame JSON-faili
DATA_DIR = "data/"
OUTFILENAME = "obj_conflicts_dataset_13032026.json"

with open(f"{DATA_DIR}{OUTFILENAME}", "w") as f:
    json.dump(obj_data, f, indent=1, ensure_ascii=False)

In [25]:
len(obj_error_data)

22

In [32]:
save_json_data_to_csv(obj_data, f"{DATA_DIR}obj_conflicts_dataset_13032026.csv")
create_example_dataframe(f"{DATA_DIR}obj_conflicts_dataset_13032026.csv", 25, f"{DATA_DIR}obj_conflicts_dataset_13032026_example_25.csv")

### advcl

In [18]:
advcl_data, advcl_error_data = get_data(f"{SOURCE_DATA_PATH}{CONFLICTS}", "advcl_has_case")

# Salvestame JSON-faili
DATA_DIR = "data/"
OUTFILENAME = "advcl_conflicts_dataset_13032026.json"

with open(f"{DATA_DIR}{OUTFILENAME}", "w") as f:
    json.dump(advcl_data, f, indent=1, ensure_ascii=False)

In [30]:
len(advcl_error_data)

0

In [33]:
save_json_data_to_csv(advcl_data, f"{DATA_DIR}advcl_conflicts_dataset_13032026.csv")
create_example_dataframe(f"{DATA_DIR}advcl_conflicts_dataset_13032026.csv", 25, f"{DATA_DIR}advcl_conflicts_dataset_13032026_example_25.csv")

### advmod

In [20]:
advmod_data, advmod_error_data = get_data(f"{SOURCE_DATA_PATH}{CONFLICTS}", "advmod_has_case")

# Salvestame JSON-faili
DATA_DIR = "data/"
OUTFILENAME = "advmod_conflicts_dataset_13032026.json"

with open(f"{DATA_DIR}{OUTFILENAME}", "w") as f:
    json.dump(advmod_data, f, indent=1, ensure_ascii=False)

In [21]:
len(advmod_error_data)

0

In [35]:
save_json_data_to_csv(advmod_data, f"{DATA_DIR}advmod_conflicts_dataset_13032026.csv")
create_example_dataframe(f"{DATA_DIR}advmod_conflicts_dataset_13032026.csv", 25, f"{DATA_DIR}advmod_conflicts_dataset_13032026_example_25.csv")

### xcomp

In [22]:
xcomp_data, xcomp_error_data = get_data(f"{SOURCE_DATA_PATH}{CONFLICTS}", "xcomp_has_case")

# Salvestame JSON-faili
DATA_DIR = "data/"
OUTFILENAME = "xcomp_conflicts_dataset_13032026.json"

with open(f"{DATA_DIR}{OUTFILENAME}", "w") as f:
    json.dump(xcomp_data, f, indent=1, ensure_ascii=False)

In [23]:
len(xcomp_error_data)

0

In [38]:
save_json_data_to_csv(xcomp_data, f"{DATA_DIR}xcomp_conflicts_dataset_13032026.csv")
create_example_dataframe(f"{DATA_DIR}xcomp_conflicts_dataset_13032026.csv", 25, f"{DATA_DIR}xcomp_conflicts_dataset_13032026_example_25.csv")

### obl

In [24]:
obl_data, obl_error_data = get_data(f"{SOURCE_DATA_PATH}{CONFLICTS}", "obl_nom")

# Salvestame JSON-faili
DATA_DIR = "data/"
OUTFILENAME = "obl_conflicts_dataset_13032026.json"

with open(f"{DATA_DIR}{OUTFILENAME}", "w") as f:
    json.dump(obl_data, f, indent=1, ensure_ascii=False)

In [40]:
len(obl_error_data)

1

In [41]:
save_json_data_to_csv(obl_data, f"{DATA_DIR}obl_conflicts_dataset_13032026.csv")
create_example_dataframe(f"{DATA_DIR}obl_conflicts_dataset_13032026.csv", 25, f"{DATA_DIR}obl_conflicts_dataset_13032026_example_25.csv")

In [ ]:
# Siin all on katsetused, siia pole vaja vaadata:

In [20]:
# Siin read, mille puhul morph_analysis kihilt ei leitud lemmat
print(result[1522])
print(result[3196])

(12635873, 'nsubj', 'gen', 2, 'Kose-Lükatile', 'Jääb Kose-Lükatile')
(15531729, 'nsubj', '', 16, 'sulama', '-4 kraadi , päike paistis ja värske lumi maas , aga neljapäeval läheb kahjuks sulaks .')


In [ ]:
# Siin read, mille puhul VabamorfAnalyzer ei leidnud lemmat
print(result[92])
print(result[1522])
print(result[1621])
print(result[2520])
print(result[3196])
print(result[6206])
print(result[7531])
print(result[7759])

(13832596, 'nsubj', 'gen', 5, 'M. Laar', 'Siin ei aita ka M. Laari 5 kl.')
(12635873, 'nsubj', 'gen', 2, 'Kose-Lükatile', 'Jääb Kose-Lükatile')
(302617, 'nsubj', 'gen', 24, 'C. Riis', 'Veetaimedest on kõige vastupidavamad sinivetikad , millised taluvad temperatuure -70 kuni -196 C. Mandritel taluvad samblikud sama madalaid temperatuure ja samblad kuni - 80 C. Riisi ja suhkruroogu kahjustab õitsemise ajal isegi +15 C temperatuur .')
(2937887, 'nsubj', 'gen', 4, 'L. Reile', 'Kilgi kinnitusel kuulus L.Reile 2000. aastate algul 14 ha kinnistuid Tallinna kesklinnas , suure osa sellest müüsid endised juhatuse liikmed võileivahinnaga maha .')
(15531729, 'nsubj', '', 16, 'sulama', '-4 kraadi , päike paistis ja värske lumi maas , aga neljapäeval läheb kahjuks sulaks .')
(367747, 'nsubj', 'gen', 1, 'B. Linne', 'B. Linde ei taha näidendit originaaliks pidada , sest “ ibsenlikud niidid ” on näha .')
(18957549, 'nsubj', 'gen', 3, 'T. Palu', "Näitena tõi T. Palu Fabrazyme Fabry tõve , Glivec'i leuke

In [16]:
test2 = "Siin ei aita ka M. Laari 5 kl."

sentence_text = Text(test2).tag_layer(["words", "sentences"])
morph_analyzer.tag(sentence_text)
sentence_text.morph_analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Siin', [{'normalized_text': 'Siin', 'lemma': 'Siin', 'root': 'Siin', 'root_tokens': ['Siin'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}, {'normalized_text': 'Siin', 'lemma': 'siin', 'root': 'siin', 'root_tokens': ['siin'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'Siin', 'lemma': 'siin', 'root': 'siin', 'root_tokens': ['siin'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}, {'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('aita', [{'normalized_text': 'aita', 'lemma': 'ait', 'root': 'ait', 'root_tokens': ['ait'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'S'}, {'normalized_text': 'aita', 'lemma': 'aitama', 'root': 'aita', 'root_tokens': ['aita'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}, {'normalized_text': 'aita', 'lemma': 'ait', 'root': 'ait', 'root_tokens': ['ait'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('ka', [{'normalized_text': 'ka', 'lemma': 'ka', 'root': 'ka', 'root_tokens': ['ka'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('M. Laari', [{'normalized_text': 'M. Laari', 'lemma': 'M. laar', 'root': 'M. _laar', 'root_tokens': ['M. ', 'laar'], 'ending': '0', 'clitic': '', 'form': 'adt', 'partofspeech': 'S'}, {'normalized_text': 'M. Laari', 'lemma': 'M. laar', 'root': 'M. _laar', 'root_tokens': ['M. ', 'laar'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'S'}, {'normalized_text': 'M. Laari', 'lemma': 'M. laar', 'root': 'M. _laar', 'root_tokens': ['M. ', 'laar'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('5', [{'normalized_text': '5', 'lemma': '5', 'root': '5', 'root_tokens': ['5'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('kl', [{'normalized_text': 'kl', 'lemma': 'kl', 'root': 'kl', 'root_tokens': ['kl'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [17]:
test2 = "Siin ei aita ka M. Laari 5 kl."

sentence_text = Text(test2).tag_layer("morph_analysis")
sentence_text.morph_analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Siin', [{'normalized_text': 'Siin', 'lemma': 'siin', 'root': 'siin', 'root_tokens': ['siin'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('ei', [{'normalized_text': 'ei', 'lemma': 'ei', 'root': 'ei', 'root_tokens': ['ei'], 'ending': '0', 'clitic': '', 'form': 'neg', 'partofspeech': 'V'}]),
Span('aita', [{'normalized_text': 'aita', 'lemma': 'aitama', 'root': 'aita', 'root_tokens': ['aita'], 'ending': '0', 'clitic': '', 'form': 'o', 'partofspeech': 'V'}]),
Span('ka', [{'normalized_text': 'ka', 'lemma': 'ka', 'root': 'ka', 'root_tokens': ['ka'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('M. Laari', [{'normalized_text': 'M. Laari', 'lemma': 'M. Laar', 'root': 'M. _Laar', 'root_tokens': ['M. ', 'Laar'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('5', [{'normalized_text': '5', 'lemma': '5', 'root': '5', 'root_tokens': ['5'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('kl', [{'normalized_text': 'kl', 'lemma': 'kl', 'root': 'kl', 'root_tokens': ['kl'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'Y'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [21]:
test3 = "Jääb Kose-Lükatile"

sentence_text = Text(test3).tag_layer("morph_analysis")
sentence_text.morph_analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('Jääb', [{'normalized_text': 'Jääb', 'lemma': 'jääma', 'root': 'jää', 'root_tokens': ['jää'], 'ending': 'b', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}]),
Span('Kose', [{'normalized_text': 'Kose', 'lemma': 'Kose', 'root': 'Kose', 'root_tokens': ['Kose'], 'ending': '0', 'clitic': '', 'form': 'sg g', 'partofspeech': 'H'}]),
Span('-', [{'normalized_text': '-', 'lemma': '-', 'root': '-', 'root_tokens': ['-'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Lükatile', [{'normalized_text': 'Lükatile', 'lemma': 'Lükatile', 'root': 'Lükatile', 'root_tokens': ['Lükatile'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}])])

In [22]:
test4 = "-4 kraadi , päike paistis ja värske lumi maas , aga neljapäeval läheb kahjuks sulaks ."

sentence_text = Text(test4).tag_layer("morph_analysis")
sentence_text.morph_analysis

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('-4', [{'normalized_text': '-4', 'lemma': '-4', 'root': '-4', 'root_tokens': ['', '4'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('kraadi', [{'normalized_text': 'kraadi', 'lemma': 'kraad', 'root': 'kraad', 'root_tokens': ['kraad'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('päike', [{'normalized_text': 'päike', 'lemma': 'päike', 'root': 'päike', 'root_tokens': ['päike'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('paistis', [{'normalized_text': 'paistis', 'lemma': 'paistma', 'root': 'paist', 'root_tokens': ['paist'], 'ending': 'is', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('ja', [{'normalized_text': 'ja', 'lemma': 'ja', 'root': 'ja', 'root_tokens': ['ja'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('värske', [{'normalized_text': 'värske', 'lemma': 'värske', 'root': 'värske', 'root_tokens': ['värske'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'A'}]),
Span('lumi', [{'normalized_text': 'lumi', 'lemma': 'lumi', 'root': 'lumi', 'root_tokens': ['lumi'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('maas', [{'normalized_text': 'maas', 'lemma': 'maas', 'root': 'maas', 'root_tokens': ['maas'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('aga', [{'normalized_text': 'aga', 'lemma': 'aga', 'root': 'aga', 'root_tokens': ['aga'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('neljapäeval', [{'normalized_text': 'neljapäeval', 'lemma': 'neljapäev', 'root': 'nelja_päev', 'root_tokens': ['nelja', 'päev'], 'ending': 'l', 'clitic': '', 'form': 'sg ad', 'partofspeech': 'S'}]),
Span('läheb', [{'normalized_text': 'läheb', 'lemma': 'minema', 'root': 'mine', 'root_tokens': ['mine'], 'ending': 'b', 'clitic': '', 'form': 'b', 'partofspeech': 'V'}]),
Span('kahjuks', [{'normalized_text': 'kahjuks', 'lemma': 'kahjuks', 'root': 'kahjuks', 'root_tokens': ['kahjuks'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('sulaks', [{'normalized_text': 'sulaks', 'lemma': 'sulama', 'root': 'sula', 'root_tokens': ['sula'], 'ending': 'ks', 'clitic': '', 'form': 'ks', 'partofspeech': 'V'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])